 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           :                 <br>
Student Number : L XXXXX         <br>
Due Date       :                 <br>
Assignment     : CA2             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [104]:
# pip installs
!pip install pytesseract pdf2image PyPDF2 pillow spacy nltk scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [105]:
#download spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 6.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [106]:
# imports
# imports
import sys
print(sys.executable)
import pymupdf

import site
print(site.getusersitepackages())

import matplotlib.pyplot as plt
# allows inline plotting
%matplotlib inline

import pytesseract
from pytesseract import Output
# regular expression
import re

from pdf2image import convert_from_path

import PyPDF2

import os
import cv2




import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# load spaCy model
nlp = spacy.load("en_core_web_sm")

from PIL import Image

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd


import PyPDF2
import pytesseract
from pdf2image import convert_from_path

/Users/shanegannon/CA2_Assignment/.venv/bin/python
/Users/shanegannon/Library/Python/3.13/lib/python/site-packages


# Support Functions


## SECTION 1 - OCR AND DOCUMENT EXTRACTION SETUP

Summary:
* These helper methods handle file type detection, OCR setup,
* PDF page conversion, direct PDF text extraction, OCR-based
* PDF extraction, and the main document extraction workflow.

In [107]:

def setup_tesseract(tesseract_path="/opt/homebrew/bin/tesseract", oem=1, psm=6):
    pytesseract.pytesseract.tesseract_cmd = tesseract_path
    custom_config = f"--oem {oem} --psm {psm}"
    return custom_config


def pdf_to_images(pdf_path, poppler_path="/opt/homebrew/bin"):
    pages = convert_from_path(pdf_path, poppler_path=poppler_path)
    return pages


def extract_text_from_image(img, config):
    text = pytesseract.image_to_string(img, config=config)
    return text


def extract_text_from_pdf_direct(pdf_path):
    all_text = []

    with open(pdf_path, "rb") as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)

        for page_number, page in enumerate(pdf_reader.pages, start=1):
            page_text = page.extract_text()

            all_text.append({
                "page": page_number,
                "text": page_text if page_text else ""
            })

    return all_text


def extract_text_from_pdf_ocr(pdf_path,
                              poppler_path="/opt/homebrew/bin",
                              tesseract_path="/opt/homebrew/bin/tesseract",
                              oem=1,
                              psm=6):
    config = setup_tesseract(
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm
    )

    pages = pdf_to_images(pdf_path, poppler_path=poppler_path)
    all_text = []

    for page_number, img in enumerate(pages, start=1):
        page_text = extract_text_from_image(img, config)

        all_text.append({
            "page": page_number,
            "text": page_text
        })

    return all_text


def extract_text_from_pdf(pdf_path,
                          poppler_path="/opt/homebrew/bin",
                          tesseract_path="/opt/homebrew/bin/tesseract",
                          oem=1,
                          psm=6,
                          min_text_length=100):
    direct_text_output = extract_text_from_pdf_direct(pdf_path)

    full_direct_text = " ".join(page_data["text"] for page_data in direct_text_output)

    if full_direct_text and len(full_direct_text.strip()) > min_text_length:
        print("Using direct PDF extraction")
        return direct_text_output, "direct_pdf"

    print("Direct extraction was poor or empty, using OCR fallback")

    ocr_text_output = extract_text_from_pdf_ocr(
        pdf_path=pdf_path,
        poppler_path=poppler_path,
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm
    )

    return ocr_text_output, "ocr_pdf"


def detect_file_type(file_path):
    file_path = file_path.lower()

    if file_path.endswith(".pdf"):
        return "pdf"
    elif file_path.endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp")):
        return "image"
    else:
        return "unsupported"


def extract_text_from_document(file_path,
                               poppler_path="/opt/homebrew/bin",
                               tesseract_path="/opt/homebrew/bin/tesseract",
                               oem=1,
                               psm=6,
                               min_text_length=100):
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        text_output, extraction_method = extract_text_from_pdf(
            pdf_path=file_path,
            poppler_path=poppler_path,
            tesseract_path=tesseract_path,
            oem=oem,
            psm=psm,
            min_text_length=min_text_length
        )
        return text_output, extraction_method

    elif file_type == "image":
        config = setup_tesseract(
            tesseract_path=tesseract_path,
            oem=oem,
            psm=psm
        )

        img = Image.open(file_path)
        image_text = extract_text_from_image(img, config)

        text_output = [{
            "page": 1,
            "text": image_text
        }]

        return text_output, "ocr_image"

    else:
        raise ValueError("Unsupported file type")



## SECTION 2 - TEXT PREPROCESSING HELPERS
Summary:
* These helper methods clean extracted text, tokenize it,
* remove stopwords, lemmatise it, convert token lists back
* nto strings, count words, and extract named entities.

In [108]:

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\-]", "", text)
    text = text.strip()

    return text


def tokenize_text(text):
    doc_object = nlp(text)

    token_list = []

    for token in doc_object:
        token_list.append(token.text)

    return token_list


def remove_stopwords(text):
    doc_object = nlp(text)

    filtered_tokens = []

    for token in doc_object:
        if not token.is_stop and not token.is_punct and not token.is_space:
            filtered_tokens.append(token.text)

    return filtered_tokens


def lemmatise_text(text):
    doc_object = nlp(text)

    lemmatised_tokens = []

    for token in doc_object:
        if not token.is_stop and not token.is_punct and not token.is_space:
            lemmatised_tokens.append(token.lemma_)

    return lemmatised_tokens


def tokens_to_string(token_list):
    text_string = " ".join(token_list)
    return text_string


def count_words(text):
    word_total = len(text.split())
    return word_total


def extract_named_entities(text):
    doc_object = nlp(text)

    entity_list = []

    for ent in doc_object.ents:
        entity_list.append((ent.text, ent.label_))

    return entity_list



## SECTION 3 - PAGE-LEVEL DATAFRAME HELPERS
Summary:
* These helper methods process extracted text page by page,
* add one row per page into documents_df,
* and print a page-level record for checking results.


In [109]:

def process_document_text(document_id, file_name, page_number, extraction_method, raw_text):
    cleaned_text = clean_text(raw_text)

    tokenised_tokens = tokenize_text(cleaned_text)
    tokenised_text = tokens_to_string(tokenised_tokens)

    no_stop_tokens = remove_stopwords(cleaned_text)
    no_stop_text = tokens_to_string(no_stop_tokens)

    lemmatised_tokens = lemmatise_text(cleaned_text)
    lemmatised_text = tokens_to_string(lemmatised_tokens)

    named_entities = extract_named_entities(cleaned_text)

    document_record = {
        "document_id": document_id,
        "file_name": file_name,
        "page_number": page_number,
        "extraction_method": extraction_method,
        "raw_text": raw_text,
        "cleaned_text": cleaned_text,
        "tokenised_text": tokenised_text,
        "no_stop_text": no_stop_text,
        "lemmatised_text": lemmatised_text,
        "raw_word_count": count_words(raw_text),
        "cleaned_word_count": count_words(cleaned_text),
        "named_entities": named_entities
    }

    return document_record


def build_document_dataframe(extracted_text_list, file_name, extraction_method="unknown", document_id=1):
    document_records = []

    for page_data in extracted_text_list:
        page_number = page_data["page"]
        raw_text = page_data["text"]

        processed_record = process_document_text(
            document_id=document_id,
            file_name=file_name,
            page_number=page_number,
            extraction_method=extraction_method,
            raw_text=raw_text
        )

        document_records.append(processed_record)

    documents_df = pd.DataFrame(document_records)

    return documents_df


def add_document_to_dataframe(documents_df, file_path, document_id,
                              poppler_path="/opt/homebrew/bin",
                              tesseract_path="/opt/homebrew/bin/tesseract",
                              oem=1,
                              psm=6,
                              min_text_length=100):
    text_output, extraction_method = extract_text_from_document(
        file_path=file_path,
        poppler_path=poppler_path,
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm,
        min_text_length=min_text_length
    )

    for page_data in text_output:
        processed_record = process_document_text(
            document_id=document_id,
            file_name=file_path,
            page_number=page_data["page"],
            extraction_method=extraction_method,
            raw_text=page_data["text"]
        )

        documents_df = pd.concat(
            [documents_df, pd.DataFrame([processed_record])],
            ignore_index=True
        )

    return documents_df


def print_document_row(documents_df, row_index):
    if documents_df.empty:
        print("Description: Dataframe is empty")
        return

    if row_index >= len(documents_df):
        print("Description: Invalid row index")
        return

    row_data = documents_df.iloc[row_index]

    print("Description: Document ID")
    print(row_data["document_id"])
    print()

    print("Description: File name")
    print(row_data["file_name"])
    print()

    print("Description: Page number")
    print(row_data["page_number"])
    print()

    print("Description: Extraction method")
    print(row_data["extraction_method"])
    print()

    print("Description: Raw text")
    print(row_data["raw_text"])
    print()

    print("Description: Cleaned text")
    print(row_data["cleaned_text"])
    print()

    print("Description: Tokenised text")
    print(row_data["tokenised_text"])
    print()

    print("Description: Text with stop words removed")
    print(row_data["no_stop_text"])
    print()

    print("Description: Lemmatised text")
    print(row_data["lemmatised_text"])
    print()

    print("Description: Raw word count")
    print(row_data["raw_word_count"])
    print()

    print("Description: Cleaned word count")
    print(row_data["cleaned_word_count"])
    print()

    print("Description: Named entities")
    print(row_data["named_entities"])



## SECTION 4 - DOCUMENT-LEVEL DATAFRAME HELPERS
Summary:
* These helper methods process a full document as one combined
* text record, add one row per full document into document_df,
* and print a document-level record for checking results.

In [110]:

def add_full_document_row_to_dataframe(document_df,
                                       document_id,
                                       file_name,
                                       extraction_method,
                                       page_count,
                                       full_raw_text,
                                       full_cleaned_text="",
                                       full_tokenised_text="",
                                       full_no_stop_text="",
                                       full_lemmatised_text="",
                                       full_named_entities="",
                                       total_raw_word_count=0,
                                       total_cleaned_word_count=0):

    new_row = {
        "document_id": document_id,
        "file_name": file_name,
        "extraction_method": extraction_method,
        "page_count": page_count,
        "full_raw_text": full_raw_text,
        "full_cleaned_text": full_cleaned_text,
        "full_tokenised_text": full_tokenised_text,
        "full_no_stop_text": full_no_stop_text,
        "full_lemmatised_text": full_lemmatised_text,
        "full_named_entities": full_named_entities,
        "total_raw_word_count": total_raw_word_count,
        "total_cleaned_word_count": total_cleaned_word_count
    }

    document_df = pd.concat([document_df, pd.DataFrame([new_row])], ignore_index=True)

    return document_df


def process_full_document_text(document_id,
                               file_name,
                               extraction_method,
                               text_output):
    full_raw_text = " ".join([page_data["text"] for page_data in text_output])

    cleaned_text = clean_text(full_raw_text)

    tokenised_tokens = tokenize_text(cleaned_text)
    full_tokenised_text = tokens_to_string(tokenised_tokens)

    no_stop_tokens = remove_stopwords(cleaned_text)
    full_no_stop_text = tokens_to_string(no_stop_tokens)

    lemmatised_tokens = lemmatise_text(cleaned_text)
    full_lemmatised_text = tokens_to_string(lemmatised_tokens)

    full_named_entities = extract_named_entities(cleaned_text)

    processed_record = {
        "document_id": document_id,
        "file_name": file_name,
        "extraction_method": extraction_method,
        "page_count": len(text_output),
        "full_raw_text": full_raw_text,
        "full_cleaned_text": cleaned_text,
        "full_tokenised_text": full_tokenised_text,
        "full_no_stop_text": full_no_stop_text,
        "full_lemmatised_text": full_lemmatised_text,
        "full_named_entities": full_named_entities,
        "total_raw_word_count": count_words(full_raw_text),
        "total_cleaned_word_count": count_words(cleaned_text)
    }

    return processed_record


def add_full_document_to_dataframe(document_df, file_path, document_id,
                                   poppler_path="/opt/homebrew/bin",
                                   tesseract_path="/opt/homebrew/bin/tesseract",
                                   oem=1,
                                   psm=6,
                                   min_text_length=100):

    text_output, extraction_method = extract_text_from_document(
        file_path=file_path,
        poppler_path=poppler_path,
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm,
        min_text_length=min_text_length
    )

    processed_record = process_full_document_text(
        document_id=document_id,
        file_name=file_path,
        extraction_method=extraction_method,
        text_output=text_output
    )

    document_df = add_full_document_row_to_dataframe(
        document_df=document_df,
        document_id=processed_record["document_id"],
        file_name=processed_record["file_name"],
        extraction_method=processed_record["extraction_method"],
        page_count=processed_record["page_count"],
        full_raw_text=processed_record["full_raw_text"],
        full_cleaned_text=processed_record["full_cleaned_text"],
        full_tokenised_text=processed_record["full_tokenised_text"],
        full_no_stop_text=processed_record["full_no_stop_text"],
        full_lemmatised_text=processed_record["full_lemmatised_text"],
        full_named_entities=processed_record["full_named_entities"],
        total_raw_word_count=processed_record["total_raw_word_count"],
        total_cleaned_word_count=processed_record["total_cleaned_word_count"]
    )

    return document_df


def print_full_document_row(document_df, row_index):
    if document_df.empty:
        print("Description: Document dataframe is empty")
        return

    if row_index >= len(document_df):
        print("Description: Invalid row index")
        return

    row_data = document_df.iloc[row_index]

    print("Description: Document ID")
    print(row_data["document_id"])
    print()

    print("Description: File name")
    print(row_data["file_name"])
    print()

    print("Description: Extraction method")
    print(row_data["extraction_method"])
    print()

    print("Description: Page count")
    print(row_data["page_count"])
    print()

    print("Description: Full raw text")
    print(row_data["full_raw_text"])
    print()

    print("Description: Full cleaned text")
    print(row_data["full_cleaned_text"])
    print()

    print("Description: Full tokenised text")
    print(row_data["full_tokenised_text"])
    print()

    print("Description: Full text with stop words removed")
    print(row_data["full_no_stop_text"])
    print()

    print("Description: Full lemmatised text")
    print(row_data["full_lemmatised_text"])
    print()

    print("Description: Full named entities")
    print(row_data["full_named_entities"])
    print()

    print("Description: Total raw word count")
    print(row_data["total_raw_word_count"])
    print()

    print("Description: Total cleaned word count")
    print(row_data["total_cleaned_word_count"])



## SECTION 5 - TF-IDF HELPERS
Summary:
These helper methods create TF-IDF features from the
full-document dataframe, identify top TF-IDF terms for
each document, and add TF-IDF summary results back into
document_df.


In [111]:

def create_tfidf_features(document_df, text_column="full_lemmatised_text", max_features=1000):
    text_data = document_df[text_column].fillna("").astype(str).tolist()

    tfidf_vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = tfidf_vectorizer.fit_transform(text_data)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    return tfidf_vectorizer, tfidf_matrix, feature_names


def build_tfidf_dataframe(tfidf_matrix, feature_names):
    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=feature_names
    )

    return tfidf_df


def get_top_tfidf_terms_for_document(tfidf_matrix, feature_names, row_index, top_n=5):
    row_vector = tfidf_matrix[row_index].toarray().flatten()
    top_indices = row_vector.argsort()[::-1][:top_n]

    top_terms = []

    for index in top_indices:
        if row_vector[index] > 0:
            top_terms.append(feature_names[index])

    return top_terms


def get_top_tfidf_terms_with_scores(tfidf_matrix, feature_names, row_index, top_n=5):
    row_vector = tfidf_matrix[row_index].toarray().flatten()
    top_indices = row_vector.argsort()[::-1][:top_n]

    top_terms_with_scores = []

    for index in top_indices:
        if row_vector[index] > 0:
            top_terms_with_scores.append((feature_names[index], row_vector[index]))

    return top_terms_with_scores


def add_top_tfidf_terms_to_document_dataframe(document_df, tfidf_matrix, feature_names, top_n=5):
    top_terms_column = []
    top_scores_column = []

    for row_index in range(len(document_df)):
        top_terms = get_top_tfidf_terms_for_document(
            tfidf_matrix=tfidf_matrix,
            feature_names=feature_names,
            row_index=row_index,
            top_n=top_n
        )

        top_terms_with_scores = get_top_tfidf_terms_with_scores(
            tfidf_matrix=tfidf_matrix,
            feature_names=feature_names,
            row_index=row_index,
            top_n=top_n
        )

        top_terms_column.append(", ".join(top_terms))
        top_scores_column.append(str(top_terms_with_scores))

    document_df["top_tfidf_terms"] = top_terms_column
    document_df["top_tfidf_terms_with_scores"] = top_scores_column

    return document_df


def print_tfidf_summary_for_document(document_df, tfidf_matrix, feature_names, row_index, top_n=5):
    if document_df.empty:
        print("Description: Document dataframe is empty")
        return

    if row_index >= len(document_df):
        print("Description: Invalid row index")
        return

    print("Description: Document ID")
    print(document_df.iloc[row_index]["document_id"])
    print()

    print("Description: File name")
    print(document_df.iloc[row_index]["file_name"])
    print()

    print("Description: Top TF-IDF terms")
    print(get_top_tfidf_terms_for_document(tfidf_matrix, feature_names, row_index, top_n))
    print()

    print("Description: Top TF-IDF terms with scores")
    print(get_top_tfidf_terms_with_scores(tfidf_matrix, feature_names, row_index, top_n))


def create_and_add_tfidf(document_df, text_column="full_lemmatised_text", max_features=1000, top_n=5):
    text_data = document_df[text_column].fillna("").astype(str).tolist()

    tfidf_vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = tfidf_vectorizer.fit_transform(text_data)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    top_terms_column = []
    top_scores_column = []

    for row_index in range(len(document_df)):
        row_vector = tfidf_matrix[row_index].toarray().flatten()
        top_indices = row_vector.argsort()[::-1][:top_n]

        top_terms = []
        top_terms_with_scores = []

        for index in top_indices:
            if row_vector[index] > 0:
                top_terms.append(feature_names[index])
                top_terms_with_scores.append((feature_names[index], row_vector[index]))

        top_terms_column.append(", ".join(top_terms))
        top_scores_column.append(str(top_terms_with_scores))

    document_df["top_tfidf_terms"] = top_terms_column
    document_df["top_tfidf_terms_with_scores"] = top_scores_column

    return document_df, tfidf_vectorizer, tfidf_matrix, feature_names

# NLP

## Text extraction

In [112]:
## code

documents_df = pd.DataFrame(columns=[
    "document_id",
    "file_name",
    "page_number",
    "extraction_method",
    "raw_text",
    "cleaned_text",
    "tokenised_text",
    "no_stop_text",
    "lemmatised_text",
    "raw_word_count",
    "cleaned_word_count",
    "named_entities"
])


# empty dataframe for one row per full document
document_df = pd.DataFrame(columns=[
    "document_id",
    "file_name",
    "extraction_method",
    "page_count",
    "full_raw_text",
    "full_cleaned_text",
    "full_tokenised_text",
    "full_no_stop_text",
    "full_lemmatised_text",
    "full_named_entities",
    "total_raw_word_count",
    "total_cleaned_word_count"
])


# file_path = "sample.pdf"
#file_path = "A_Midsummer_Night.pdf"
#file_path = "HistoricalDoc2.jpg"
file_path = "85629964.png"

document_id = 1

documents_df = add_document_to_dataframe(
    documents_df,
    file_path,
    1
)

# document-level dataframe
document_df = add_full_document_to_dataframe(
    document_df,
    file_path,
    document_id
)

document_df, tfidf_vectorizer, tfidf_matrix, feature_names = create_and_add_tfidf(
    document_df=document_df,
    text_column="full_lemmatised_text",
    max_features=1000,
    top_n=5
)

print("Description: Document dataframe with top TF-IDF terms")
print(document_df[["document_id", "file_name", "top_tfidf_terms"]])

#print page version
print_document_row(documents_df,0)

#print full document version
print_full_document_row(document_df, 0)


print("Description: Document dataframe with top TF-IDF terms")
print(document_df[["document_id", "file_name", "top_tfidf_terms"]])

Description: Document dataframe with top TF-IDF terms
  document_id     file_name                    top_tfidf_terms
0           1  85629964.png  expense, cat, date, sale, variety
Description: Document ID
1

Description: File name
85629964.png

Description: Page number
1

Description: Extraction method
ocr_image

Description: Raw text
a c c
CIGARETTE REPORT FORM

YEAR: NO. PER PACK:

BRAND NAME:

VAR. DESC: (SEE EXPLANATION)

VARIETY UNIT SALES: VARIETY DOLLAR SALES:

CIG. LENGTH: FILTER LENGTH:

FILTER TYPE: FLAVORING: OVERWRAP: PACK TYPE:

AST MANUFACT. DATE: 1ST SALES DATE: LAST SOLD DATE:

YEARLY SUMMARY:

TAR: NICOTINE: CARBON MONO:

ADVERTISING EXPENDITURES (SEE EXPLANATION)

CAT~A-EXPENSES: CAT-B-EXPENSES : CAT-C-EXPENSES:

CAT-D-EXPENSES: CAT-E-EXPENSES: CAT-P-EXPENSES:

CAT-G-EXPENSES: CAT-H-EXPENSES : CAT-1-EXPENSES :

CAT-J-EXPENSES: CAT-K-EXPENSES: CAT-L-EXPENSES:

CAT-M-EXPENSES : CAT-N-EXPENSES :

TOTAL ADVERTISING EXPENDITURES:
x
a
EA
8
3
3
cd
&

i


Description: Cleaned

In [113]:
print("Description: Number of page rows for document 1")
print(len(documents_df[documents_df["document_id"] == 1]))

print("Description: Number of full document rows for document 1")
print(len(document_df[document_df["document_id"] == 1]))

Description: Number of page rows for document 1
1
Description: Number of full document rows for document 1
1


# Vision

## Sub Heading 1

In [114]:
# code here...

# Multi-modal

## Sub Heading 1

In [115]:
# code here

# Final Output

In [116]:
# code